# 在 Colab 中打开
<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ai-agents-the-definitive-guide/blob/main/CH10/ch10_memory_no_memory.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Notebook 说明


本 Notebook 演示了在 LangGraph 聊天机器人中，启用与不启用检查点式记忆（checkpointed memory）时的差异。

这个图只包含一个聊天机器人节点，它通过兼容 OpenRouter 的 `ChatOpenAI` 接口，把当前消息历史发送给大语言模型（LLM）。同一个图会被编译成两个版本：一个不使用 checkpointer，另一个使用 `InMemorySaver` 作为 checkpointer。

第一次运行时，每条用户消息都会被图独立处理，因此不同调用之间不会保留用户姓名。第二次运行时，两次调用使用同一个 `thread_id`，LangGraph 因而可以从记忆中恢复之前的会话状态。于是，聊天机器人能够利用上一轮提供的信息回答后续问题。

这个示例说明了线程级检查点（thread-level checkpointing）在 Agent 记忆中的作用，也说明了为什么在构建对话式或多轮 Agent 系统时，必须显式配置持久状态。


In [ ]:
import os
from dotenv import load_dotenv


load_dotenv()

OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')

In [ ]:
!pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 3.0 MB/s eta 0:00:00


In [ ]:
from typing import Annotated, TypedDict
import operator

from langchain_core.messages import AnyMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver


llm = ChatOpenAI(model="gpt-5.4-nano", base_url="https://openrouter.ai/api/v1", temperature=0, api_key=OPENROUTER_API_KEY)


In [ ]:
# pip install -U langgraph langchain openai



# ---------------------------
# 定义状态
# ---------------------------
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]


# ---------------------------
# 定义节点
# ---------------------------
def chatbot(state: AgentState) -> AgentState:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


# ---------------------------
# 构建图
# ---------------------------
builder = StateGraph(AgentState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)


# 版本 A：不使用记忆
graph_without_memory = builder.compile()

# 版本 B：使用记忆
checkpointer = InMemorySaver()
graph_with_memory = builder.compile(checkpointer=checkpointer)


# ---------------------------
# 用于打印输出的辅助函数
# ---------------------------
def show_last_ai_message(result: dict) -> None:
    last_message = result["messages"][-1]
    print(f"AI: {last_message.content}\n")


# ---------------------------
# 不使用记忆运行
# ---------------------------
print("WITHOUT MEMORY")
print("=" * 60)

result = graph_without_memory.invoke(
    {
        "messages": [
            HumanMessage(content="My name is Nicole. I am writing a chapter about memory in AI agents. You don't need to give me tips, just ackknowledge it.")
        ]
    }
)
show_last_ai_message(result)

result = graph_without_memory.invoke(
    {
        "messages": [
            HumanMessage(content="What is my name?")
        ]
    }
)
show_last_ai_message(result)


# ---------------------------
# 使用记忆运行
# ---------------------------
print("WITH MEMORY")
print("=" * 60)

config = {"configurable": {"thread_id": "demo-thread-1"}}

result = graph_with_memory.invoke(
    {
        "messages": [
            HumanMessage(content="My name is Nicole. I am writing a chapter about memory in AI agents.")
        ]
    },
    config=config,
)
show_last_ai_message(result)

result = graph_with_memory.invoke(
    {
        "messages": [
            HumanMessage(content="What is my name?")
        ]
    },
    config=config,
)
show_last_ai_message(result)

WITHOUT MEMORY
AI: Hi Nicole — acknowledged. Good luck writing your chapter on memory in AI agents.

AI: I don’t know your name from this chat. If you tell me what you’d like me to call you, I’ll use it.

WITH MEMORY
AI: Hi Nicole—nice to meet you! If you’re writing a chapter on memory in AI agents, I can help you shape the content, outline the chapter, or draft sections.

A few quick questions to tailor it:
1. **Audience level:** is this for general readers, undergraduate, or graduate/professional?
2. **Scope:** do you mean *agent memory* broadly (retrieval, long-term/short-term, planning), or mainly *neural memory* (transformers, external memory, RNN/attention)?
3. **Angle:** are you emphasizing **technical mechanisms** (e.g., RAG, vector DBs, episodic memory) or **systems/engineering** (latency, scaling, evaluation), or both?
4. **Length target:** roughly how many pages/words?

Meanwhile, here are a couple of strong chapter section structures you can choose from:

### Option A: Conc